<a href="https://colab.research.google.com/github/WJKoh13/FarmPestManagementAI/blob/main/VGG16_Pytorch_VGG16_PROPest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install pytorch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/WJKoh13/FarmPestManagementAI.git /content/FarmPestManagementAI
%cd /content/FarmPestManagementAI
!pip install -q pyyaml pandas scikit-learn tqdm   # torch/torchvision/matplotlib preinstalled


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/FarmPestManagementAI' already exists and is not an empty directory.
/content/FarmPestManagementAI


In [ ]:
# Verify the contents of the cloned repository
!ls -F /content/FarmPestManagementAI

configs/	 IP102_v1.1/  requirements.txt	src/
data_manifests/  README.md    scripts/


In [22]:
# 1. Create the classes.txt in the format expected by the setup script
# The script seems to look for a mapping that includes specific class indices.
# We will write the 102 class placeholders.
import os

class_file_path = '/content/FarmPestManagementAI/IP102_v1.1/Classification/classes.txt'
os.makedirs(os.path.dirname(class_file_path), exist_ok=True)

with open(class_file_path, 'w') as f:
    for i in range(102):
        f.write(f'{i} class_{i}\n')

# 2. Run the setup script. We point it to the nested folder where the images actually are
!python scripts/setup_data.py --tar /content/ip102_v1.1.tar

[skip] dataset already extracted at /content/FarmPestManagementAI/IP102_v1.1/Classification/ip102_v1.1
[OK] train.csv: 4318 images (expected 4318)
        per class: 0=669, 1=292, 2=631, 3=302, 4=303, 5=500, 6=535, 7=331, 8=513, 9=242
[OK] validation.csv: 721 images (expected 721)
        per class: 0=111, 1=48, 2=106, 3=50, 4=51, 5=83, 6=90, 7=56, 8=86, 9=40
[OK] test.csv: 2166 images (expected 2166)
        per class: 0=335, 1=147, 2=316, 3=152, 4=152, 5=251, 6=268, 7=166, 8=257, 9=122
[ok] wrote selected_classes.json

All manifest counts match the expected totals.


In [20]:
import torch
import torch.nn as nn
from torchvision import models

def create_vgg16_model(num_classes=102):
    # Load pre-trained VGG16
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

    # Freeze training for all layers (optional: you can unfreeze if you want to fine-tune everything)
    for param in model.parameters():
        param.requires_grad = False

    # Modify the classifier (last layer)
    # VGG16 classifier has 7 layers, model.classifier[6] is the final Linear layer
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)

    return model

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_vgg16_model(num_classes=102).to(device)

print(f"VGG16 model created and moved to: {device}")
print(model.classifier)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 81.4MB/s]


VGG16 model created and moved to: cpu
Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=102, bias=True)
)


In [29]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

class IP102Dataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # The CSV file contains relative paths like 'images/03692.jpg'
        # We combine this with the Classification folder path
        img_name = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.data.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transforms for VGG16
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Corrected path to the folder containing the 'images' subdirectory
base_path = '/content/FarmPestManagementAI'
img_path = '/content/FarmPestManagementAI/IP102_v1.1/Classification/ip102_v1.1'

train_dataset = IP102Dataset(csv_file=f'{base_path}/data_manifests/train.csv', img_dir=img_path, transform=transform)
val_dataset = IP102Dataset(csv_file=f'{base_path}/data_manifests/validation.csv', img_dir=img_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"DataLoaders re-created: {len(train_dataset)} training images, {len(val_dataset)} validation images.")

DataLoaders re-created: 4318 training images, 721 validation images.


In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Hyperparameters
num_epochs = 10
learning_rate = 0.001

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Use tqdm for progress bar
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss / len(train_loader)})

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Accuracy: {100 * correct / total:.2f}%')

print("Training complete.")

Epoch 1/10:   4%|▍         | 6/135 [02:33<53:17, 24.79s/it, loss=0.15]

In [ ]:
# Check the extraction result to find the classes.txt and image folders
!ls -R /content/FarmPestManagementAI/IP102_v1.1/

/content/FarmPestManagementAI/IP102_v1.1/:
Classification

/content/FarmPestManagementAI/IP102_v1.1/Classification:
classes.txt  ip102_v1.1

/content/FarmPestManagementAI/IP102_v1.1/Classification/ip102_v1.1:
images	test.txt  train.txt  val.txt

/content/FarmPestManagementAI/IP102_v1.1/Classification/ip102_v1.1/images:
00000.jpg  10746.jpg  21492.jpg  32238.jpg  42984.jpg  53730.jpg  64476.jpg
00001.jpg  10747.jpg  21493.jpg  32239.jpg  42985.jpg  53731.jpg  64477.jpg
00002.jpg  10748.jpg  21494.jpg  32240.jpg  42986.jpg  53732.jpg  64478.jpg
00003.jpg  10749.jpg  21495.jpg  32241.jpg  42987.jpg  53733.jpg  64479.jpg
00004.jpg  10750.jpg  21496.jpg  32242.jpg  42988.jpg  53734.jpg  64480.jpg
00005.jpg  10751.jpg  21497.jpg  32243.jpg  42989.jpg  53735.jpg  64481.jpg
00006.jpg  10752.jpg  21498.jpg  32244.jpg  42990.jpg  53736.jpg  64482.jpg
00007.jpg  10753.jpg  21499.jpg  32245.jpg  42991.jpg  53737.jpg  64483.jpg
00008.jpg  10754.jpg  21500.jpg  32246.jpg  42992.jpg  53738.jpg  64484